# 🤖 AI สำหรับ AIoT Cucumber Classroom (รันบน Colab)
โน้ตบุ๊กนี้เปิด LLM ขนาดเล็ก (Qwen2.5 3B ผ่าน Ollama) บน Colab แล้วสร้างลิงก์สาธารณะให้ gateway ของคุณเรียกใช้

**วิธีใช้:** เลือก Runtime → Change runtime type → **T4 GPU** แล้วรันเซลล์ทีละอัน (หรือ Run all)
เมื่อได้ลิงก์ `https://....trycloudflare.com` ให้ใส่ในไฟล์ `.env` ของ gateway เป็น `OLLAMA_URL=...` แล้วรีสตาร์ท gateway

In [ ]:
# 1) ติดตั้ง Ollama และเปิดเซิร์ฟเวอร์
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import subprocess, time
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print("ollama server started")

In [ ]:
# 2) ดาวน์โหลดโมเดล (ครั้งแรกใช้เวลาสักครู่)
MODEL = "qwen2.5:3b"
!ollama pull {MODEL}

In [ ]:
# 3) ทดสอบถามโมเดลแบบเดียวกับที่ gateway จะทำ
import json, urllib.request
payload = {"model": MODEL, "stream": False, "messages": [
    {"role": "system", "content": "ตอบเป็นภาษาไทย สั้นๆ ใช้เฉพาะข้อมูลที่ให้"},
    {"role": "user", "content": "ข้อมูล: อุณหภูมิ 33.1 °C ความชื้น 62%\nคำถาม: ตอนนี้ร้อนไหม"}]}
req = urllib.request.Request("http://localhost:11434/api/chat", json.dumps(payload).encode(),
                             {"Content-Type": "application/json"})
print(json.load(urllib.request.urlopen(req))["message"]["content"])

In [ ]:
# 4) สร้างลิงก์สาธารณะด้วย cloudflared (ไม่ต้องสมัครบัญชี)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
import re, subprocess
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:11434"],
                          stderr=subprocess.PIPE, text=True)
for line in tunnel.stderr:
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m:
        print("OLLAMA_URL=" + m.group(0))
        break
print("เก็บเซลล์นี้ให้รันค้างไว้ตลอดการใช้งาน")

### หมายเหตุ
- ลิงก์เปลี่ยนทุกครั้งที่รันใหม่ และ Colab ฟรีจะหยุดเองเมื่อไม่มีการใช้งาน
- ถ้า gateway ติดต่อ LLM ไม่ได้ ระบบจะตอบด้วยกฎง่ายๆ แทนโดยอัตโนมัติ (ไม่พัง)
- ใครเปิดลิงก์นี้ได้ก็ใช้โมเดลของคุณได้ อย่าแชร์ลิงก์ต่อสาธารณะ